#### Venv
Create a venv using python -m venv venv
activate the venv
and install the requirements.txt

### importing  required Libs

In [40]:
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

### Loading the dataset

In [41]:
from datasets import load_dataset
dataset = load_dataset(
"csv",
data_files="trainV2.csv"
)


In [42]:
dataset["train"][:5]

{'Text': ['நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்',
  'உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங்கமா தாண்டி இருக்கும்',
  'கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்து வையுங்கள்',
  'ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ ஒனக்கு இந்த சைஸ் போதுமா?',
  'ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷோ நடத்தி டெவலப் பண்றீயா'],
 'Class': ['Non-Abusive', 'Abusive', 'Non-Abusive', 'Abusive', 'Non-Abusive']}

Normalizing the data by
> Removing white space at end 
> Converting the case 

In [43]:
label2id = {
    "Non-Abusive": 0,
    "Abusive": 1
}
def encode_labels(example):
    label = example["Class"].strip()
    if label.lower() == "abusive":
        example["Class"] = label2id["Abusive"]
    elif label.lower() == "non-abusive":
        example["Class"] = label2id["Non-Abusive"]
    else:
        # Fallback: try direct lookup
        example["Class"] = label2id[label]
    return example
dataset = dataset.map(encode_labels)

In [44]:
dataset = dataset.rename_column("Text", "text")
dataset = dataset.rename_column("Class", "label")

In [45]:
# Convert labels from string to integer
# Required because:
# 1. PyTorch loss functions (CrossEntropyLoss) expect integer class indices, not strings
# 2. Model outputs are indexed by integers (class 0, class 1, etc.)
# 3. Tensor operations require numeric types, not strings
# 4. Integer operations are much faster than string comparisons
def convert_label(example):
    example["label"] = int(example["label"])
    return example

dataset = dataset.map(convert_label)

Map: 100%|██████████| 3652/3652 [00:00<00:00, 41758.94 examples/s]


In [46]:
dataset["train"][:5]

{'text': ['நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்',
  'உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங்கமா தாண்டி இருக்கும்',
  'கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்து வையுங்கள்',
  'ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ ஒனக்கு இந்த சைஸ் போதுமா?',
  'ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷோ நடத்தி டெவலப் பண்றீயா'],
 'label': ['0', '1', '0', '1', '0']}